# Unsupervised model candidates

In [23]:
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import precision_score, f1_score
from pyod.models.hbos import HBOS
from pyod.models.copod import COPOD
from pyod.models.ecod import ECOD
from sklearn.ensemble import IsolationForest
import optuna


class UnsupervisedAnomalyDetector:
    def __init__(self, contamination=0.05):
        self.contamination = contamination
        self.models = {}
        self.best_params = {}
        self.scores = {}

    def optimize_isolation_forest(self, X, trial):
        """Optimize IsolationForest hyperparameters"""
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_samples': trial.suggest_float('max_samples', 0.1, 1.0),
            'max_features': trial.suggest_float('max_features', 0.1, 1.0),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
            'contamination': self.contamination,
            'random_state': 42
        }

        model = IsolationForest(**params)
        model.fit(X)
        scores = -model.score_samples(X)
        return self._calculate_objective(scores)

    def optimize_lof(self, X, trial):
        """Optimize LocalOutlierFactor hyperparameters"""
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 5, 50),
            'leaf_size': trial.suggest_int('leaf_size', 10, 100),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan']),
            'contamination': self.contamination
        }

        model = LocalOutlierFactor(**params)
        scores = -model.fit_predict(X)
        return self._calculate_objective(scores)

    def optimize_hbos(self, X, trial):
        """Optimize HBOS hyperparameters"""
        params = {
            'n_bins': trial.suggest_int('n_bins', 5, 50),
            'alpha': trial.suggest_float('alpha', 0.1, 1.0),
            'tol': trial.suggest_float('tol', 0.1, 0.5),
            'contamination': self.contamination
        }

        model = HBOS(**params)
        model.fit(X)
        scores = model.decision_function(X)
        return self._calculate_objective(scores)

    def _calculate_objective(self, scores):
        """Calculate objective value for optimization"""
        threshold = np.percentile(scores, (1 - self.contamination) * 100)
        predictions = (scores > threshold).astype(int)
        return np.mean(predictions == (scores > np.median(scores)))

    def optimize_models(self, X, n_trials=100):
        """Optimize all models using Optuna"""
        optimization_funcs = {
            'IsolationForest': self.optimize_isolation_forest,
            'LOF': self.optimize_lof,
            'HBOS': self.optimize_hbos
        }

        for name, optimize_func in optimization_funcs.items():
            print(f"\nOptimizing {name}...")
            study = optuna.create_study(direction='maximize')
            study.optimize(lambda trial: optimize_func(X, trial), n_trials=n_trials)
            self.best_params[name] = study.best_params
            print(f"Best parameters for {name}: {study.best_params}")

    def fit_predict(self, X, y=None):
        """Fit and predict with all models"""
        # Initialize models with best parameters
        self.models['IsolationForest'] = IsolationForest(
            **self.best_params['IsolationForest'],
            contamination=self.contamination
        )

        self.models['LOF'] = LocalOutlierFactor(
            **self.best_params['LOF'],
            contamination=self.contamination
        )

        self.models['HBOS'] = HBOS(
            **self.best_params['HBOS'],
            contamination=self.contamination
        )

        # Add other state-of-the-art models with default parameters
        self.models['COPOD'] = COPOD(contamination=self.contamination)
        self.models['ECOD'] = ECOD(contamination=self.contamination)
        self.models['EllipticEnvelope'] = EllipticEnvelope(
            contamination=self.contamination,
            random_state=42
        )

        # Fit and predict with each model
        predictions = {}
        scores = {}

        for name, model in self.models.items():
            print(f"\nFitting {name}...")
            if isinstance(model, LocalOutlierFactor):
                predictions[name] = model.fit_predict(X)
                scores[name] = model.negative_outlier_factor_
            else:
                model.fit(X)
                predictions[name] = model.predict(X)
                scores[name] = model.decision_function(X) if hasattr(model, 'decision_function') \
                    else model.score_samples(X)

        return predictions, scores

    def evaluate(self, predictions, y_true):
        """Evaluate model performance if true labels are available"""
        results = {}

        for name, y_pred in predictions.items():
            # Convert predictions to binary (1 for anomaly, 0 for normal)
            y_pred_binary = (y_pred == -1).astype(int)

            results[name] = {
                'precision': precision_score(y_true, y_pred_binary),
                'recall': recall_score(y_true, y_pred_binary),
                'f1': f1_score(y_true, y_pred_binary)
            }

        return pd.DataFrame(results).T


def main():
    # Load data
    print("Loading data...")
    # df = pd.read_csv('raw_transactions.csv')
    df = df_treated.copy()

    # Prepare features
    X = df.drop(['infraction', 'event_created_at', 'merchant_id'], axis=1)
    y = df['infraction']  # Only used for evaluation

    # Initialize detector
    detector = UnsupervisedAnomalyDetector(contamination=0.05)  # 5% anomalies

    # Optimize models
    detector.optimize_models(X)

    # Fit and predict
    predictions, scores = detector.fit_predict(X)

    # Evaluate results
    results = detector.evaluate(predictions, y)
    print("\nModel Performance:")
    print(results)

    # Save results
    results.to_csv('unsupervised_model_results.csv')

    # Create DataFrame with all scores and predictions
    results_df = pd.DataFrame(index=df.index)

    for name in detector.models.keys():
        results_df[f'{name}_score'] = scores[name]
        results_df[f'{name}_prediction'] = predictions[name]

    # Add ensemble score (average of all normalized scores)
    normalized_scores = pd.DataFrame(scores).apply(lambda x: (x - x.mean()) / x.std())
    results_df['ensemble_score'] = normalized_scores.mean(axis=1)

    # Add original data and true labels
    results_df = pd.concat([df, results_df], axis=1)

    # Save detailed results
    results_df.to_csv('unsupervised_detailed_results.csv')

    return detector, results, results_df


if __name__ == "__main__":
    detector, results, results_df = main()

Loading data...


[I 2025-02-12 14:59:40,406] A new study created in memory with name: no-name-1f4c5f71-b7b3-450d-96a2-7667e801c3a3



Optimizing IsolationForest...


[I 2025-02-12 15:00:16,612] Trial 0 finished with value: 0.5500012601251052 and parameters: {'n_estimators': 146, 'max_samples': 0.27379766624480584, 'max_features': 0.10005217783088406, 'bootstrap': True}. Best is trial 0 with value: 0.5500012601251052.
[I 2025-02-12 15:01:13,572] Trial 1 finished with value: 0.5500012601251052 and parameters: {'n_estimators': 207, 'max_samples': 0.11104780973117924, 'max_features': 0.4230177401232913, 'bootstrap': True}. Best is trial 0 with value: 0.5500012601251052.
[I 2025-02-12 15:01:56,134] Trial 2 finished with value: 0.5500012601251052 and parameters: {'n_estimators': 87, 'max_samples': 0.7582915345351396, 'max_features': 0.59146532098031, 'bootstrap': False}. Best is trial 0 with value: 0.5500012601251052.
[I 2025-02-12 15:02:46,748] Trial 3 finished with value: 0.5500012601251052 and parameters: {'n_estimators': 153, 'max_samples': 0.9182328241557108, 'max_features': 0.22331511552283323, 'bootstrap': True}. Best is trial 0 with value: 0.5500

Best parameters for IsolationForest: {'n_estimators': 149, 'max_samples': 0.6314716080873201, 'max_features': 0.45015174184417606, 'bootstrap': True}

Optimizing LOF...


NameError: name 'LocalOutlierFactor' is not defined

In [ ]:
import numpy as np
import pandas as pd

# For model selection
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import recall_score, classification_report

# Unsupervised anomaly detection models
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# =============================================================================
# Define a custom scoring function for unsupervised models.
# Since these models output -1 for anomalies and 1 for inliers,
# we convert predictions so that:
#    -1 becomes 1 (suspicious) and 1 becomes 0 (normal),
# and then compute recall (i.e. fraction of true frauds that are flagged).
# =============================================================================
from sklearn.metrics import make_scorer


def unsupervised_recall_scorer(estimator, X, y):
    # For LOF and One-Class SVM, ensure that the model has been fitted with novelty=True if needed.
    y_pred = estimator.predict(X)
    # Convert: anomalies (-1) -> 1, inliers (1) -> 0
    y_pred_bin = np.where(y_pred == -1, 1, 0)
    return recall_score(y, y_pred_bin, zero_division=0)


unsupervised_recall = make_scorer(unsupervised_recall_scorer)

# =============================================================================
# Load data (raw, without scaling)
# =============================================================================
print("Loading data...")
# df = pd.read_csv('raw_transactions.csv')
df = df_treated.copy()

# Prepare features by dropping columns not used for prediction.
X = df.drop(['infraction', 'event_created_at', 'merchant_id'], axis=1)
y = df['infraction']

print(f"Data shape: {X.shape}")
print(f"Number of fraud cases: {sum(y == 1)}")
print(f"Fraud ratio: {sum(y == 1) / len(y):.4%}")

# Train-test split (stratify so that test set maintains the original distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# =============================================================================
# Define unsupervised candidate models with parameter grids.
# Note: For LOF, we must set novelty=True so that it can be used on new (test) data.
# =============================================================================
unsupervised_models = {
    "Isolation Forest": {
        "estimator": IsolationForest(random_state=42),
        "param_grid": {
            "n_estimators": [100, 200],
            "max_samples": [0.5, 0.75, 1.0],
            "contamination": [0.0005, 0.001, 0.005]
        }
    },
    "Local Outlier Factor": {
        # Set novelty=True to enable predicting on unseen data.
        "estimator": LocalOutlierFactor(novelty=True),
        "param_grid": {
            "n_neighbors": [20, 30, 50],
            "contamination": [0.0005, 0.001, 0.005]
        }
    },
    "One-Class SVM": {
        "estimator": OneClassSVM(kernel='rbf'),
        "param_grid": {
            "nu": [0.0005, 0.001, 0.005],  # nu is an upper bound on the fraction of training errors
            "gamma": ['scale', 0.001, 0.01, 0.1]
        }
    }
}

# =============================================================================
# Hyperparameter tuning for each unsupervised model using GridSearchCV.
# We use our custom unsupervised_recall scorer.
# =============================================================================
best_unsupervised_models = {}

print("\n--- Hyperparameter Tuning for Unsupervised Models ---")
for model_name, model_info in unsupervised_models.items():
    print(f"\n----- {model_name} -----")
    grid = GridSearchCV(
        estimator=model_info["estimator"],
        param_grid=model_info["param_grid"],
        scoring=unsupervised_recall,
        cv=5,  # 5-fold cross-validation on the training set
        n_jobs=-1,
        verbose=1
    )
    # Fit grid search on the training set (which remains unaltered)
    grid.fit(X_train, y_train)
    print(f"Best parameters for {model_name}: {grid.best_params_}")
    print(f"Best CV Recall for {model_name}: {grid.best_score_:.4f}")
    best_unsupervised_models[model_name] = grid.best_estimator_

# =============================================================================
# Evaluate each candidate unsupervised model on the test set.
# =============================================================================
print("\n--- Evaluation on Test Set for Unsupervised Models ---")
unsupervised_results = {}

for model_name, model in best_unsupervised_models.items():
    print(f"\nEvaluating {model_name}...")
    # Use predict on test set; convert -1 to 1 (suspicious) and 1 to 0 (normal)
    y_pred = model.predict(X_test)
    y_pred_bin = np.where(y_pred == -1, 1, 0)

    recall_val = recall_score(y_test, y_pred_bin, zero_division=0)
    print(classification_report(y_test, y_pred_bin, zero_division=0))

    unsupervised_results[model_name] = recall_val
    print(f"Test Recall for {model_name}: {recall_val:.4f}")

# Select the best unsupervised model (based on recall on test set)
best_unsupervised_model_name = max(unsupervised_results, key=unsupervised_results.get)
print(
    f"\nSelected Unsupervised Model: {best_unsupervised_model_name} with Test Recall = {unsupervised_results[best_unsupervised_model_name]:.4f}")

# =============================================================================
# With the winning unsupervised model, generate anomaly scores and rank transactions.
# =============================================================================
winning_model = best_unsupervised_models[best_unsupervised_model_name]

# For models that support decision_function (e.g., Isolation Forest, One-Class SVM, LOF with novelty=True)
# lower scores (more negative) indicate higher anomaly.
if hasattr(winning_model, "decision_function"):
    anomaly_scores = winning_model.decision_function(X)
else:
    # If decision_function is not available, we can use negative of predict_proba if available,
    # or simply use the raw predictions.
    anomaly_scores = -winning_model.predict(X)

# Append anomaly scores to the original dataframe (use the full dataset for ranking)
df['anomaly_score'] = anomaly_scores

# Rank transactions: most suspicious first (i.e. smallest decision function score)
df_ranked = df.sort_values('anomaly_score', ascending=True)

# For instance, select the top 100 most suspicious transactions:
top_100_anomalies = df_ranked.head(100)
print("\nTop 100 most suspicious transactions:")
print(top_100_anomalies[['anomaly_score', 'infraction']])

# The ranked list can now be used by your fraud team to prioritize cases for human evaluation.
